In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [22]:
# ==============================
# 1. Baseline settings
# ==============================
DATASETS = ["100_GT.csv", "300_GT.csv"]
THRESHOLD_GRID_STEP = 0.01

# Baseline definition used for final prediction per dataset
BASELINE_SIMILARITY = "combined_similarity"

OUTPUT_DIR = Path("similarity")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_SUMMARY_CSV = OUTPUT_DIR / "similarity_baseline_comparison.csv"
OUTPUT_BEST_THRESHOLD_CSV = OUTPUT_DIR / "similarity_best_thresholds_by_accuracy.csv"

In [3]:
def load_and_prepare(csv_path: str) -> tuple[pd.DataFrame, str]:
    df = pd.read_csv(csv_path)
    feature2_col = "APP Features 2" if "APP Features 2" in df.columns else "App Features 2"
    text_cols = ["APP Features 1", "Review 1", feature2_col, "Review 2"]

    for col in text_cols:
        df[col] = df[col].fillna("").astype(str).str.strip()

    df["Annotation"] = pd.to_numeric(df["Annotation"], errors="coerce").fillna(0).astype(int)
    return df, feature2_col

Expected columns:
"APP Features 1", "Review 1", "APP Features 2", "Review 2", "Annotation"

In [ ]:
# This notebook now uses load_and_prepare() inside the dataset loop.
# No single-dataframe preprocessing is needed in this cell.

In [ ]:
# Text cleaning is handled inside load_and_prepare(csv_path).

In [ ]:
# Annotation conversion is handled inside load_and_prepare(csv_path).

In [4]:
# ==============================
# 2. Similarity + threshold helpers
# ==============================
def pairwise_tfidf_cosine(text1, text2):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text1, text2])
    sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    return float(sim)

In [5]:
def evaluate_threshold(y_true, sims, threshold):
    y_pred = (sims >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "threshold": float(threshold),
        "accuracy": float(acc),
        "f1": float(f1),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [6]:
def pick_best_by_accuracy(rows):
    # Tie-break: higher F1, then threshold closest to 0.5 for stability.
    return max(rows, key=lambda r: (r["accuracy"], r["f1"], -abs(r["threshold"] - 0.5)))

In [7]:
def compute_similarities(df: pd.DataFrame, feature2_col: str) -> pd.DataFrame:
    feature_sims = []
    review_sims = []
    combined_sims = []

    for _, row in df.iterrows():
        feature_1 = row["APP Features 1"]
        review_1 = row["Review 1"]
        feature_2 = row[feature2_col]
        review_2 = row["Review 2"]

        feature_sim = pairwise_tfidf_cosine(feature_1, feature_2)
        review_sim = pairwise_tfidf_cosine(review_1, review_2)

        left_text = feature_1 + " " + review_1
        right_text = feature_2 + " " + review_2
        combined_sim = pairwise_tfidf_cosine(left_text, right_text)

        feature_sims.append(feature_sim)
        review_sims.append(review_sim)
        combined_sims.append(combined_sim)

    out = df.copy()
    out["feature_similarity"] = feature_sims
    out["review_similarity"] = review_sims
    out["combined_similarity"] = combined_sims
    return out

In [8]:
def threshold_search(df_sim: pd.DataFrame, step: float = THRESHOLD_GRID_STEP) -> pd.DataFrame:
    thresholds = np.round(np.arange(0.0, 1.0 + step, step), 4)
    y_true = df_sim["Annotation"].to_numpy()

    all_eval_rows = []
    for sim_col in ["feature_similarity", "review_similarity", "combined_similarity"]:
        sims = df_sim[sim_col].to_numpy()
        for t in thresholds:
            res = evaluate_threshold(y_true, sims, t)
            res["similarity_type"] = sim_col
            all_eval_rows.append(res)

    return pd.DataFrame(all_eval_rows)

In [9]:
def best_thresholds_by_accuracy(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for sim_col in ["feature_similarity", "review_similarity", "combined_similarity"]:
        subset = eval_df[eval_df["similarity_type"] == sim_col].to_dict(orient="records")
        best_acc = pick_best_by_accuracy(subset)
        rows.append({
            "similarity_type": sim_col,
            "best_threshold": best_acc["threshold"],
            "best_accuracy": best_acc["accuracy"],
            "best_f1": best_acc["f1"],
            "tn": best_acc["tn"],
            "fp": best_acc["fp"],
            "fn": best_acc["fn"],
            "tp": best_acc["tp"],
        })
    return pd.DataFrame(rows)

In [10]:
def baseline_metrics_from_best(df_sim: pd.DataFrame, best_df: pd.DataFrame, baseline_col: str) -> dict:
    y_true = df_sim["Annotation"].to_numpy()
    best_row = best_df[best_df["similarity_type"] == baseline_col].iloc[0]
    th = float(best_row["best_threshold"])

    y_pred = (df_sim[baseline_col].to_numpy() >= th).astype(int)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "baseline_similarity": baseline_col,
        "threshold": th,
        "accuracy": float(acc),
        "f1": float(f1),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [23]:
all_baseline_rows = []
all_best_rows = []
all_paths_to_best = []

for csv_path in DATASETS:
    df_raw, feature2_col = load_and_prepare(csv_path)
    df_sim = compute_similarities(df_raw, feature2_col)
    eval_df = threshold_search(df_sim)
    best_df = best_thresholds_by_accuracy(eval_df)

    stem = Path(csv_path).stem
    pair_out = OUTPUT_DIR / f"pair_similarity_results_{stem}.csv"
    threshold_out = OUTPUT_DIR / f"threshold_search_results_{stem}.csv"
    best_out = OUTPUT_DIR / f"best_thresholds_by_accuracy_{stem}.csv"

    df_sim.to_csv(pair_out, index=False)
    eval_df.to_csv(threshold_out, index=False)
    best_df.to_csv(best_out, index=False)

    baseline_row = baseline_metrics_from_best(df_sim, best_df, BASELINE_SIMILARITY)
    baseline_row["dataset"] = csv_path
    all_baseline_rows.append(baseline_row)

    best_df_with_dataset = best_df.copy()
    best_df_with_dataset["dataset"] = csv_path
    all_best_rows.append(best_df_with_dataset)

    best_th = float(best_df.loc[best_df["similarity_type"] == BASELINE_SIMILARITY, "best_threshold"].iloc[0])
    path_to_best = eval_df[
        (eval_df["similarity_type"] == BASELINE_SIMILARITY)
        & (eval_df["threshold"] <= best_th)
    ].copy().sort_values("threshold")
    path_to_best["dataset"] = csv_path
    path_to_best["baseline_similarity"] = BASELINE_SIMILARITY
    path_to_best["is_best_threshold"] = (path_to_best["threshold"] == best_th).astype(int)
    all_paths_to_best.append(path_to_best)

    path_out = OUTPUT_DIR / f"accuracy_path_to_best_{stem}.csv"
    path_to_best.to_csv(path_out, index=False)

    print(f"\nDataset: {csv_path}")
    print(f"Saved: {pair_out}")
    print(f"Saved: {threshold_out}")
    print(f"Saved: {best_out}")
    print(f"Saved: {path_out}")

baseline_summary_df = pd.DataFrame(all_baseline_rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
best_thresholds_all_df = pd.concat(all_best_rows, ignore_index=True)
accuracy_path_to_best_df = pd.concat(all_paths_to_best, ignore_index=True)

baseline_summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False)
best_thresholds_all_df.to_csv(OUTPUT_BEST_THRESHOLD_CSV, index=False)

print(f"\nSaved: {OUTPUT_SUMMARY_CSV}")
print(f"Saved: {OUTPUT_BEST_THRESHOLD_CSV}")


Dataset: 100_GT.csv
Saved: similarity\pair_similarity_results_100_GT.csv
Saved: similarity\threshold_search_results_100_GT.csv
Saved: similarity\best_thresholds_by_accuracy_100_GT.csv
Saved: similarity\accuracy_path_to_best_100_GT.csv

Dataset: 300_GT.csv
Saved: similarity\pair_similarity_results_300_GT.csv
Saved: similarity\threshold_search_results_300_GT.csv
Saved: similarity\best_thresholds_by_accuracy_300_GT.csv
Saved: similarity\accuracy_path_to_best_300_GT.csv

Saved: similarity\similarity_baseline_comparison.csv
Saved: similarity\similarity_best_thresholds_by_accuracy.csv


In [12]:
print("Baseline comparison:")
display(
    baseline_summary_df[
        ["dataset", "baseline_similarity", "threshold", "accuracy", "f1", "tn", "fp", "fn", "tp"]
    ]
)

Baseline comparison:


,dataset,baseline_similarity,threshold,accuracy,f1,tn,fp,fn,tp
0,300_GT.csv,combined_similarity,0.27,0.66,0.578512,128,35,67,70
1,100_GT.csv,combined_similarity,0.99,0.64,0.000000,64,0,36,0


In [13]:
print("Best threshold by accuracy for each similarity type and dataset:")
display(
    best_thresholds_all_df[
        ["dataset", "similarity_type", "best_threshold", "best_accuracy", "best_f1", "tn", "fp", "fn", "tp"]
    ].sort_values(["dataset", "similarity_type"])
)

Best threshold by accuracy for each similarity type and dataset:


,dataset,similarity_type,best_threshold,best_accuracy,best_f1,tn,fp,fn,tp
2,100_GT.csv,combined_similarity,0.99,0.640000,0.000000,64,0,36,0
0,100_GT.csv,feature_similarity,0.61,0.680000,0.272727,62,2,30,6
1,100_GT.csv,review_similarity,1.00,0.550000,0.000000,55,9,36,0
5,300_GT.csv,combined_similarity,0.27,0.660000,0.578512,128,35,67,70
3,300_GT.csv,feature_similarity,0.61,0.790000,0.709677,160,3,60,77
4,300_GT.csv,review_similarity,0.15,0.616667,0.552529,114,49,66,71


In [14]:
print("Accuracy at each threshold until best accuracy is reached (baseline similarity):")
display(
    accuracy_path_to_best_df[
        ["dataset", "baseline_similarity", "threshold", "accuracy", "f1", "is_best_threshold"]
    ].sort_values(["dataset", "threshold"])
)

Accuracy at each threshold until best accuracy is reached (baseline similarity):


,dataset,baseline_similarity,threshold,accuracy,f1,is_best_threshold
0,100_GT.csv,combined_similarity,0.00,0.360000,0.529412,0
1,100_GT.csv,combined_similarity,0.01,0.370000,0.519084,0
2,100_GT.csv,combined_similarity,0.02,0.380000,0.523077,0
3,100_GT.csv,combined_similarity,0.03,0.370000,0.511628,0
4,100_GT.csv,combined_similarity,0.04,0.400000,0.516129,0
...,...,...,...,...,...,...
123,300_GT.csv,combined_similarity,0.23,0.636667,0.594796,0
124,300_GT.csv,combined_similarity,0.24,0.646667,0.598485,0
125,300_GT.csv,combined_similarity,0.25,0.640000,0.578125,0
126,300_GT.csv,combined_similarity,0.26,0.646667,0.572581,0


In [15]:
for ds in DATASETS:
    print(f"\nThreshold-to-best accuracy path for: {ds}")
    view_df = accuracy_path_to_best_df[accuracy_path_to_best_df["dataset"] == ds][
        ["threshold", "accuracy", "f1", "is_best_threshold"]
    ].sort_values("threshold")
    display(view_df)


Threshold-to-best accuracy path for: 100_GT.csv


,threshold,accuracy,f1,is_best_threshold
0,0.00,0.36,0.529412,0
1,0.01,0.37,0.519084,0
2,0.02,0.38,0.523077,0
3,0.03,0.37,0.511628,0
4,0.04,0.40,0.516129,0
...,...,...,...,...
95,0.95,0.56,0.000000,0
96,0.96,0.58,0.000000,0
97,0.97,0.60,0.000000,0
98,0.98,0.62,0.000000,0



Threshold-to-best accuracy path for: 300_GT.csv


,threshold,accuracy,f1,is_best_threshold
100,0.00,0.456667,0.627002,0
101,0.01,0.456667,0.627002,0
102,0.02,0.456667,0.627002,0
103,0.03,0.456667,0.627002,0
104,0.04,0.456667,0.627002,0
105,0.05,0.456667,0.627002,0
106,0.06,0.463333,0.629885,0
107,0.07,0.466667,0.631336,0
108,0.08,0.470000,0.631090,0
109,0.09,0.483333,0.635294,0


In [16]:
print("Best baseline model settings by dataset:")
for _, r in baseline_summary_df.iterrows():
    print(
        f"{r['dataset']}: baseline={r['baseline_similarity']} "
        f"threshold={r['threshold']:.2f} accuracy={r['accuracy']:.4f} f1={r['f1']:.4f} "
        f"confusion=[[{int(r['tn'])}, {int(r['fp'])}], [{int(r['fn'])}, {int(r['tp'])}]]"
    )

Best baseline model settings by dataset:
300_GT.csv: baseline=combined_similarity threshold=0.27 accuracy=0.6600 f1=0.5785 confusion=[[128, 35], [67, 70]]
100_GT.csv: baseline=combined_similarity threshold=0.99 accuracy=0.6400 f1=0.0000 confusion=[[64, 0], [36, 0]]


In [24]:
# Keep a compact threshold-only CSV for quick plotting/reporting
threshold_accuracy_only_df = accuracy_path_to_best_df[
    ["dataset", "baseline_similarity", "threshold", "accuracy", "f1", "is_best_threshold"]
].copy()
threshold_accuracy_path = OUTPUT_DIR / "threshold_accuracy_until_best.csv"
threshold_accuracy_only_df.to_csv(threshold_accuracy_path, index=False)
print("Saved:", threshold_accuracy_path)

Saved: similarity\threshold_accuracy_until_best.csv


In [25]:
lines = []
lines.append("TF-IDF Cosine Similarity Baseline Report")
lines.append(f"Datasets: {', '.join(DATASETS)}")
lines.append(f"Baseline similarity: {BASELINE_SIMILARITY}")
lines.append("")

In [26]:
for _, r in baseline_summary_df.iterrows():
    lines.append(f"Dataset: {r['dataset']}")
    lines.append(f"  Threshold: {r['threshold']:.2f}")
    lines.append(f"  Accuracy: {r['accuracy']:.4f}")
    lines.append(f"  F1: {r['f1']:.4f}")
    lines.append(f"  Confusion [[TN, FP], [FN, TP]]: [[{int(r['tn'])}, {int(r['fp'])}], [{int(r['fn'])}, {int(r['tp'])}]]")

    path_df = accuracy_path_to_best_df[
        accuracy_path_to_best_df["dataset"] == r["dataset"]
    ][["threshold", "accuracy", "f1", "is_best_threshold"]].sort_values("threshold")

    lines.append("  Accuracy by threshold until best:")
    for _, p in path_df.iterrows():
        marker = " <= best" if int(p["is_best_threshold"]) == 1 else ""
        lines.append(
            f"    th={float(p['threshold']):.2f} acc={float(p['accuracy']):.4f} f1={float(p['f1']):.4f}{marker}"
        )
    lines.append("")

In [27]:
report_path = OUTPUT_DIR / "similarity_threshold_metrics.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

In [28]:
print("Saved:", OUTPUT_SUMMARY_CSV)
print("Saved:", OUTPUT_BEST_THRESHOLD_CSV)
print("Saved:", threshold_accuracy_path)
print("Saved:", report_path)

Saved: similarity\similarity_baseline_comparison.csv
Saved: similarity\similarity_best_thresholds_by_accuracy.csv
Saved: similarity\threshold_accuracy_until_best.csv
Saved: similarity\similarity_threshold_metrics.txt
